# Analisis de conductores con SQLite

Cargar los CSV en SQLite, calcular los totales por conductor mediante SQL y exportar el resumen y el grafico de los diez conductores con mas millas.


In [ ]:
from contextlib import closing
from pathlib import Path
import sqlite3

import matplotlib.pyplot as plt
from matplotlib.ticker import StrMethodFormatter
import pandas as pd

activity_dir = next(
    path for path in (
        Path.cwd(), Path.cwd().parent, Path.cwd() / "P104_drivers_sqlite"
    )
    if path.name == "P104_drivers_sqlite" and (path / "data/drivers.csv").is_file()
)
submission_dir = activity_dir / "submission"
submission_dir.mkdir(parents=True, exist_ok=True)


In [ ]:
# Pandas se usa para leer los CSV; la agregacion y el ranking se hacen en SQL.
drivers = pd.read_csv(activity_dir / "data/drivers.csv")
timesheet = pd.read_csv(activity_dir / "data/timesheet.csv").rename(
    columns={
        "driverId": "driver_id",
        "hours-logged": "hours_logged",
        "miles-logged": "miles_logged",
    }
)
drivers = drivers.rename(columns={"driverId": "driver_id"})


In [ ]:
# La base en memoria se reconstruye en cada ejecucion y se cierra al terminar.
with closing(sqlite3.connect(":memory:")) as connection:
    drivers[["driver_id", "name"]].to_sql(
        "drivers", connection, index=False, if_exists="replace"
    )
    timesheet.to_sql("timesheet", connection, index=False, if_exists="replace")
    connection.execute(
        "CREATE UNIQUE INDEX driver_id_unique ON drivers(driver_id)"
    )
    connection.execute("""
        CREATE VIEW driver_summary AS
        SELECT
            t.driver_id,
            SUM(t.hours_logged) AS hours_logged,
            SUM(t.miles_logged) AS miles_logged,
            d.name
        FROM timesheet AS t
        INNER JOIN drivers AS d ON t.driver_id = d.driver_id
        GROUP BY t.driver_id, d.name
    """)
    summary = pd.read_sql_query(
        "SELECT * FROM driver_summary ORDER BY driver_id", connection
    )
    top10 = pd.read_sql_query(
        """SELECT * FROM driver_summary
           ORDER BY miles_logged DESC, driver_id ASC
           LIMIT 10""",
        connection,
    )

summary.head()


In [ ]:
summary.to_csv(submission_dir / "summary.csv", index=False)
top10


In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(top10["name"], top10["miles_logged"], color="tab:blue", alpha=0.8)
ax.invert_yaxis()
ax.set_title("Top 10 conductores por millas recorridas")
ax.set_xlabel("Millas acumuladas")
ax.set_ylabel("Conductor")
ax.xaxis.set_major_formatter(StrMethodFormatter("{x:,.0f}"))
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
fig.savefig(submission_dir / "top10_drivers.png", dpi=150, bbox_inches="tight")
plt.show()
plt.close(fig)
